# Exercise 24: User Activity for the Past 30 Days I

**Source:** [LeetCode 1141](https://leetcode.com/problems/user-activity-for-the-past-30-days-i/description/?envType=study-plan-v2&envId=top-sql-50) (Easy).

Table: `Activity`
```
+---------------+---------+
| Column Name   | Type    |
+---------------+---------+
| user_id       | int     |
| session_id    | int     |
| activity_date | date    |
| activity_type | enum    |
+---------------+---------+
```
- The table has no primary key and may have duplicate row.
- `user_id` is the user running `session_id`.
- `activity_date` denotes the the date `YYYY-MM-DD` of the activity.
- The `activity_type` is an ENUM type with categories `open_session`, `end_session`, `scroll_down`, `send_message`.

**Task.**
Find the number of daily active users `active_users` from the 30 days up to `2019-07-27`.

Connect to the database `postgres` using the username `postgres`.

In [1]:
%load_ext sql
%sql postgresql+psycopg://postgres@localhost:5432/postgres

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

Connecting to 'postgresql+psycopg://postgres@localhost:5432/postgres'

### Create new database for the exercise

Make sure first that a database with the same name is removed in case it already exists.

In [2]:
%sql DROP DATABASE IF EXISTS exercise_24 WITH (FORCE);
%sql CREATE DATABASE exercise_24;

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

++
||
++
++

Connect to the database.

*Caution.* No semicolon at the end otherwise it belongs to the name of the database.

In [3]:
%sql postgresql+psycopg://postgres@localhost:5432/exercise_24 --alias exercise_24

Connecting and switching to connection 'exercise_24'

#### Fill the database with content
Since there is an `ENUM` type, it needs to be defined before.

In [4]:
%sql CREATE TYPE act_type AS ENUM ('open_session', 'end_session', 'scroll_down', 'send_message'); 

Running query in 'exercise_24'

++
||
++
++

Now fill the table content.

In [6]:
%%sql
DROP TABLE IF EXISTS Activity;

CREATE TABLE Activity (
    user_id INTEGER NOT NULL,
    session_id INTEGER NOT NULL,
    activity_date DATE NOT NULL,
    activity_type act_type NOT NULL
);

INSERT INTO Activity (user_id, session_id, activity_date, activity_type)
VALUES
(1, 1, '2019-07-20', 'open_session'),
(1, 1, '2019-07-20', 'scroll_down'),
(1, 1, '2019-07-20', 'end_session'),
(2, 4, '2019-07-20', 'open_session'),
(2, 4, '2019-07-21', 'send_message'),
(2, 4, '2019-07-21', 'end_session'),
(3, 2, '2019-07-21', 'open_session'),
(3, 2, '2019-07-21', 'send_message'),
(3, 2, '2019-07-21', 'end_session'),
(4, 3, '2019-06-25', 'open_session'),
(4, 3, '2019-06-25', 'end_session');

SELECT * FROM Activity;

Running query in 'exercise_24'

11 rows affected.

11 rows affected.

user_id,session_id,activity_date,activity_type
1,1,2019-07-20,open_session
1,1,2019-07-20,scroll_down
1,1,2019-07-20,end_session
2,4,2019-07-20,open_session
2,4,2019-07-21,send_message
2,4,2019-07-21,end_session
3,2,2019-07-21,open_session
3,2,2019-07-21,send_message
3,2,2019-07-21,end_session
4,3,2019-06-25,open_session


#### Solve the exercise

**Task.**
Find the number of daily active users `active_users` from the 30 days up to `2019-07-27`.

In [7]:
%%sql
SELECT
    activity_date AS day, COUNT(DISTINCT user_id) AS active_users
FROM
    Activity
GROUP BY
    activity_date
HAVING
    activity_date >= DATE_TRUNC('day', '2019-07-27'::DATE - INTERVAL '29 days') AND activity_date <= '2019-07-27';

Running query in 'exercise_24'

2 rows affected.

day,active_users
2019-07-20,2
2019-07-21,2


#### Explanation

- By grouping over the `activity_date` and counting the `DISTINCT` `user_id`, one obtains a table which returns all `active_users` for every date recorded in the table `Activity`.
- To make sure to take only the period between `2019-06-29` and `2019-07-27` into account subtract `2019-07-27` with 29 days using `'2019-07-27'::DATE - INTERVAL '29 days'` (the snippet `::DATE` makes sure that `2019-07-27` is read as a date).
Note that `DATE_TRUNC('day', _)` makes sure that only the date (without hours, minutes, etc.) is returned.

#### Close connection

In [8]:
%sql --close exercise_24
%sql postgresql+psycopg://postgres@localhost:5432/postgres
%sql DROP DATABASE IF EXISTS exercise_24;

Switching to connection 'postgresql+psycopg://postgres@localhost:5432/postgres'

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

++
||
++
++

**Well done!** (approved by Leet)